# Gradient Descent

California housing regression. Learning rate sweep and batch/SGD/mini-batch comparison.

## Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
from sklearn.model_selection import train_test_split

data_df = pd.read_csv("housing.csv")
data_df['total_bedrooms'] = data_df['total_bedrooms'].fillna(data_df['total_bedrooms'].median())
data_encoded = pd.get_dummies(data_df, columns=['ocean_proximity'], dtype=float)

data = data_encoded.to_numpy()
target_col = data_encoded.columns.get_loc('median_house_value')
feature_cols = [i for i in range(data.shape[1]) if i != target_col]
x_data = data[:, feature_cols]
y = data[:, target_col]
feature_names = [data_encoded.columns[i] for i in feature_cols]

mu = x_data.mean(axis=0)
sigma = x_data.std(axis=0)
sigma[sigma == 0] = 1
r_data = (x_data - mu) / sigma
r_data = np.c_[np.ones(r_data.shape[0]), r_data]

X_train, X_test, y_train, y_test = train_test_split(
    r_data, y, test_size=0.33, shuffle=True, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Learning rate sweep

In [ ]:
alphas = [0.001, 0.01, 0.05, 0.1, 0.3]
iterations = 500

fig, axs = plt.subplots(1, 2, figsize=(20, 6))

for alpha in alphas:
    theta = np.random.randn(X_train.shape[1]) * 0.01
    mse_history = []

    for i in range(iterations):
        y_hat = X_train @ theta
        mse = (1 / X_train.shape[0]) * np.sum((y_hat - y_train) ** 2)
        mse_history.append(mse)
        delta = (2 / X_train.shape[0]) * (X_train.T @ (y_hat - y_train))
        theta = theta - alpha * delta

    axs[0].plot(mse_history, label=f'α={alpha}')
    axs[1].plot(mse_history[:50], label=f'α={alpha}')

axs[0].set_xlabel('Iteration')
axs[0].set_ylabel('MSE')
axs[0].set_title('Full')
axs[0].legend()
axs[0].set_ylim(0, 3e10)

axs[1].set_xlabel('Iteration')
axs[1].set_ylabel('MSE')
axs[1].set_title('First 50')
axs[1].legend()
axs[1].set_ylim(0, 3e10)

plt.tight_layout()
plt.show()

## Batch vs SGD vs mini-batch

In [ ]:
np.random.seed(42)
alpha = 0.05
iterations = 300

theta_batch = np.random.randn(X_train.shape[1]) * 0.01
theta_sgd = theta_batch.copy()
theta_mini = theta_batch.copy()

batch_mse = []
sgd_mse = []
mini_mse = []
batch_size = 64

for i in range(iterations):
    y_hat = X_train @ theta_batch
    batch_mse.append((1 / X_train.shape[0]) * np.sum((y_hat - y_train) ** 2))
    delta = (2 / X_train.shape[0]) * (X_train.T @ (y_hat - y_train))
    theta_batch = theta_batch - alpha * delta

    idx = np.random.randint(0, X_train.shape[0])
    x_i = X_train[idx:idx+1]
    y_i = y_train[idx:idx+1]
    y_hat_i = x_i @ theta_sgd
    sgd_mse.append((1 / X_train.shape[0]) * np.sum((X_train @ theta_sgd - y_train) ** 2))
    delta_sgd = 2 * (x_i.T @ (y_hat_i - y_i))
    theta_sgd = theta_sgd - alpha * delta_sgd

    indices = np.random.choice(X_train.shape[0], batch_size, replace=False)
    x_mb = X_train[indices]
    y_mb = y_train[indices]
    y_hat_mb = x_mb @ theta_mini
    mini_mse.append((1 / X_train.shape[0]) * np.sum((X_train @ theta_mini - y_train) ** 2))
    delta_mb = (2 / batch_size) * (x_mb.T @ (y_hat_mb - y_mb))
    theta_mini = theta_mini - alpha * delta_mb

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(batch_mse, label='Batch', linewidth=2)
ax.plot(sgd_mse, label='SGD', alpha=0.7)
ax.plot(mini_mse, label='Mini-batch (64)', alpha=0.7)
ax.set_xlabel('Iteration')
ax.set_ylabel('MSE')
ax.legend()
ax.set_ylim(0, 2e10)
plt.tight_layout()
plt.show()

## Test predictions

In [ ]:
variants = {
    'Batch': theta_batch,
    'SGD': theta_sgd,
    'Mini-batch': theta_mini
}

fig, axs = plt.subplots(3, 1, figsize=(18, 18))

for i, (name, theta_v) in enumerate(variants.items()):
    y_pred = X_test @ theta_v

    mse = np.round((1 / X_test.shape[0]) * np.sum((y_pred - y_test) ** 2), 2)
    mae = np.round((1 / X_test.shape[0]) * np.sum(np.abs(y_pred - y_test)), 2)
    r2 = np.round(sk.metrics.r2_score(y_test, y_pred), 4)

    axs[i].scatter(range(X_test.shape[0]), y_pred, s=5, alpha=0.3, label='Predicted')
    axs[i].scatter(range(X_test.shape[0]), y_test, s=5, alpha=0.3, label='Actual')
    axs[i].set_title(name)
    axs[i].set_xlabel('Sample')
    axs[i].set_ylabel('House Value ($)')
    axs[i].legend()

    errors = f"MSE: {mse}\nMAE: {mae}\nR2: {r2}"
    axs[i].text(0.01, 0.95, errors, transform=axs[i].transAxes, verticalalignment='top',
               fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Error distribution

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(20, 5))

for i, (name, theta_v) in enumerate(variants.items()):
    errors = X_test @ theta_v - y_test
    axs[i].hist(errors, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axs[i].set_title(name)
    axs[i].set_xlabel('Error ($)')
    axs[i].set_ylabel('Frequency')
    axs[i].axvline(x=0, color='red', linestyle='--', linewidth=2)

plt.tight_layout()
plt.show()